## Imports

In [ ]:
import os
import typing as T
import warnings
from datetime import datetime
from pathlib import Path

import hydra
import lightning as L
import rootutils
from hydra import compose
from hydra.core.global_hydra import GlobalHydra
from lightning.pytorch.loggers import CSVLogger, Logger, MLFlowLogger, WandbLogger
from omegaconf import OmegaConf

from ntt.utils import RankedLogger, print_config_tree

# Setup root directory
root = rootutils.setup_root("..", indicator=".project-root", pythonpath=True)

# Setup logger to only log from rank zero. NOTE: In the notebook we just use
#  `print`. During train/eval runs, logging is enhanced by Hydra
#  (`Hydra logging documentation <https://hydra.cc/docs/tutorials/basic/running_your_app/logging/>`_).
log = RankedLogger(__name__, rank_zero_only=True)

## Setup

### Globals

In [ ]:
# Set checkpoint path
SERVER_NAME = "null"  # use default with `'null'`
EXPERIMENT_OVERRIDE = "null"  # use default with `'null'`
RUN_NAME = f"playground-{datetime.now().strftime('%m-%d-%Y-%H_%M')}"
JOB = "train"

### Config

In [ ]:
# Clear any existing Hydra initialization
GlobalHydra.instance().clear()
# Initialize Hydra + compose the config
hydra.initialize(config_path="../config", version_base="1.1")
cfg = compose(
    config_name="main",
    overrides=[
        f"paths.run_name={RUN_NAME}",
        #     f'local={SERVER_NAME}',
        #     f'experiment={EXPERIMENT_OVERRIDE}',
        f"paths.output_dir=${{paths.log_dir}}/{JOB}/${{paths.run_name}}",
        # "logger=mlflow",
        # f"~hydra=null",
    ],
)

# Create output dir if it doesn't exist
output_dir = Path(cfg.paths.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

# Extras: filter warnings + print config tree
if cfg.ignore_warnings:
    print("Disabling python warnings!")
    warnings.filterwarnings("ignore")
if cfg.print_config:
    print_config_tree(cfg, resolve=False)

### Experiment Tracking

In [ ]:
# Instantiate logger
logger: Logger = hydra.utils.instantiate(cfg.logger)
if isinstance(logger, MLFlowLogger):
    print(
        f"Logging to MLFlow ({
            os.path.join(
                logger._tracking_uri,
                '#',
                'experiments',
                logger.experiment_id,
                'runs',
                logger.run_id,  # type: ignore
            )
        }"
    )
elif isinstance(logger, WandbLogger):
    # Initialize wandb experiment
    logger.experiment  # noqa: B018
elif isinstance(logger, CSVLogger):
    print(f"Logging to CSV ({logger.log_dir})")

# Log config hyperparameters to experiment tracker
cfg_container: dict[str, T.Any] = OmegaConf.to_container(cfg, resolve=False)  # pyright: ignore[reportAssignmentType]
logger.log_hyperparams(cfg_container)

### Set Seed

In [ ]:
L.seed_everything(cfg.seed, workers=True)

## Data

In [ ]:
hydra.utils.instantiate(cfg.model)

In [ ]:
OmegaConf.to_container(cfg.model, resolve=True)